# Laboratorium 1 - analiza koszykowa

## Przygotowanie

 * pobierz i wypakuj dataset: https://kaggle.com/datasets/rashikrahmanpritom/groceries-dataset-for-market-basket-analysismba?resource=download&select=basket.csv
   * alternatywnie, pobierz plik `basket.csv` z Teamsów
 * [opcjonalnie] Utwórz wirtualne środowisko
 `python3 -m venv ./recsyslab1`
 * zainstaluj potrzebne biblioteki:
 `pip install more-itertools`

## Część 1. - przygotowanie danych

In [28]:
# importujemy wszystkie potrzebne pakiety

import random
from tqdm import tqdm
from typing import NamedTuple
from more_itertools import powerset
import numpy as np

In [5]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("rashikrahmanpritom/groceries-dataset-for-market-basket-analysismba")

# print("Path to dataset files:", path)

In [2]:
# definiujemy stałe

PATH = './basket.csv'
EPSILON = 0.001
N = 20          # liczba rekomendacji zwracanych przez recommend()
K = [3, 5, 10]  # wartości k używane przy ewaluacji
N_EVAL = 200    # liczba koszyków użytych do ewaluacji
SEED = 42

In [3]:
# wczytujemy dane o koszykach

def read_baskets(path: str) -> list[set[str]]:
    with open(path) as f:
        raw = f.read()
    baskets = [
        {item.lower() for item in row.split(',') if item}
        for row in raw.split('\n')[1:]
        if row
    ]
    return baskets


def unique_products(baskets: list[set[str]]) -> list[str]:
    products: set[str] = set()
    for basket in baskets:
        products.update(basket)
    return sorted(products)


baskets = read_baskets(PATH)
products = unique_products(baskets)
print(f"Wczytano {len(baskets)} koszyków i {len(products)} unikalnych produktów.")

Wczytano 14963 koszyków i 167 unikalnych produktów.


## Część 2. - obliczanie wskaźników

In [4]:
test_dict = {frozenset({1,2}): None, frozenset({2,3,4}): None}
print("start")
print(" " * 2, end="")
print(" ".join([f"{set(a)}" for a in test_dict.keys()]))
print("end")

start
  {1, 2} {2, 3, 4}
end


In [5]:
class BasketBranch:
    def __init__(self, basket: frozenset | None = None):
        self.basket = basket # which basket this branch represents (root is None)
        self.duplicates = 0 # how many times an identical basket has been added to this branch
        self.top_baskets: dict[frozenset, BasketBranch] = {}
        # self.descendants = 0
    
    def add_basket(self, new_basket: frozenset):
        # self.descendants += 1
        for basket, branch in self.top_baskets.items():
            if new_basket == basket:
                branch.duplicates += 1
                break
            if new_basket.issubset(basket):
                branch.add_basket(new_basket=new_basket)
                break
            if basket.issubset(new_basket):
                new_branch = BasketBranch(basket=new_basket)
                new_branch.duplicates = 1
                new_branch.top_baskets[basket] = branch
                # new_branch.descendants = branch.descendants + 1
                self.top_baskets[new_basket] = new_branch
                del self.top_baskets[basket]
                break
        else:
            new_branch = BasketBranch(basket=new_basket)
            new_branch.duplicates = 1
            self.top_baskets[new_basket] = new_branch

    def __str__(self, depth=0):
        for basket, branch in self.top_baskets.items():
            print(" " * depth + f"{set(basket)} (duplicates: {branch.duplicates})")
            branch.__str__(depth=depth+2)
        return str(self.basket)

In [ ]:
set1 = set({1,2,3})
valid_products = [1, 2, 3, 4, 5]
set1 in set(valid_products), set1, set(valid_products)

(False, {1, 2, 3}, {1, 2, 3, 4, 5})

In [ ]:
# obliczamy strukturę danych (słownik) przechowującą wszystkie interesujące wartości `support`

Supports = dict[frozenset, float]

def get_supports(
    baskets: list[set[str]],
    all_products: list[str],
    epsilon: float,
) -> Supports:
    supports: Supports = {}
    skipped: set[frozenset] = set()
    
    def traverse_tree(tree: BasketBranch, target_basket: frozenset):
        local_sum = 0
        for basket, branch in tree.top_baskets.items():
            if target_basket.issubset(basket):
                local_sum += traverse_tree(branch, target_basket)
        return local_sum + tree.duplicates
    
    support_tree_root = BasketBranch()
    for og_basket in baskets:
        support_tree_root.add_basket(new_basket=frozenset(og_basket))
        
    
    for og_basket in tqdm(baskets):
        for product in all_products:
            for basket in set((frozenset(og_basket), frozenset(og_basket | {product}))):
                if supports.get(basket, None) is not None or basket in skipped:
                    continue
                
                support = traverse_tree(support_tree_root, basket) / len(baskets)
                if support > epsilon:
                    supports[basket] = support
                else:
                    skipped.add(basket)
                    

    return supports




In [102]:
# test basket tree

test_baskets = [{1,2}, {1}, {1,2,3}, {2,3}, {1,2}, {1,2,3,4}, {3,4,5,6}, {3,4}, {7}, {7}]

test_root = BasketBranch()
for test_basket in test_baskets:
    test_root.add_basket(new_basket=frozenset(test_basket))
print(test_root)

test_supports = get_supports(test_baskets, [1], EPSILON)
test_supports

{1, 2, 3, 4} (duplicates: 1)
  {1, 2, 3} (duplicates: 1)
    {1, 2} (duplicates: 2)
      {1} (duplicates: 1)
    {2, 3} (duplicates: 1)
  {3, 4} (duplicates: 1)
{3, 4, 5, 6} (duplicates: 1)
{7} (duplicates: 2)
None


100%|██████████| 10/10 [00:00<?, ?it/s]


{frozenset({1, 2}): 0.4,
 frozenset({1}): 0.5,
 frozenset({1, 2, 3}): 0.2,
 frozenset({2, 3}): 0.3,
 frozenset({1, 2, 3, 4}): 0.1,
 frozenset({3, 4, 5, 6}): 0.1,
 frozenset({3, 4}): 0.3,
 frozenset({1, 3, 4}): 0.1,
 frozenset({7}): 0.2}

In [ ]:
supports = get_supports(baskets, products, EPSILON)
supports

100%|██████████| 14963/14963 [09:08<00:00, 27.28it/s]


{frozenset({'rolls/buns', 'sausage', 'whole milk'}): 0.0011361358016440553,
 frozenset({'soda', 'whole milk'}): 0.011628684087415625,
 frozenset({'other vegetables', 'soda', 'whole milk'}): 0.0011361358016440553,
 frozenset({'rolls/buns', 'soda', 'whole milk'}): 0.0010024727661565194,
 frozenset({'sausage', 'soda', 'whole milk'}): 0.0010693042839002875,
 frozenset({'curd', 'frankfurter'}): 0.0013366303548753592,
 frozenset({'butter', 'whole milk'}): 0.004678206242063757,
 frozenset({'frozen vegetables', 'other vegetables'}): 0.003141081333957094,
 frozenset({'sugar', 'tropical fruit'}): 0.0010024727661565194,
 frozenset({'rolls/buns'}): 0.10940319454654815,
 frozenset({'beef', 'rolls/buns'}): 0.001603956425850431,
 frozenset({'berries', 'rolls/buns'}): 0.001670787943594199,
 frozenset({'beverages', 'rolls/buns'}): 0.0012029673193878234,
 frozenset({'bottled beer', 'rolls/buns'}): 0.00394305954688231,
 frozenset({'bottled water', 'rolls/buns'}): 0.004678206242063757,
 frozenset({'brown 

In [ ]:
# definiujemy funkcje obliczające support, confidence i lift

def support(supports: Supports, products: set[str]) -> float:
    return supports.get(frozenset(products), 0.0)


def confidence(
    supports: Supports,
    prior: set[str],
    following: set[str],
) -> float:
    prior_support = support(supports, prior)

    if prior_support == 0:
        return 0.0

    return support(supports, prior | following) / prior_support


def lift(
    supports: Supports,
    prior: set[str],
    following: set[str],
) -> float:
    prior_following_support = support(supports, prior) * support(supports, following)
    
    if prior_following_support == 0:
        return 0.0
    
    return support(supports, prior | following) / prior_following_support

In [66]:
print(support(supports, {'whole milk', 'rolls/buns'}))
print(confidence(supports, {'whole milk', 'rolls/buns'}, {'yogurt'}))
print(lift(supports, {'whole milk', 'rolls/buns'}, {'yogurt'}))

0.013900955690703735
0.09615384615384616
1.119649805447471


## Część 3. - generowanie rekomendacji

In [76]:
# wyznaczamy listę potencjalnych rekomendacji
# rekomendowane artykuły powinny mieć lift > 1 i jak najwyższe confidence

class Candidate(NamedTuple):
    item: str
    subbasket: set[str]
    confidence: float
    lift: float


def generate_next_product_candidates(
    basket: set[str],
    products: list[str],
    supports: Supports,
) -> list[Candidate]:
    candidates = []

    for product in products:
        if product in basket:
            continue

        following = {product}
        for prior in powerset(basket):
            prior = set(prior)
            product_confidence = confidence(supports, prior, following)
            product_lift = lift(supports, prior, following)

            if product_confidence > 0 and product_lift > 1:
                candidates.append(
                    Candidate(
                        item=product,
                        subbasket=set(basket),
                        confidence=product_confidence,
                        lift=product_lift,
                    )
                )

    return sorted(
        candidates,
        key=lambda candidate: candidate.confidence,
        reverse=True,
    )


In [92]:
# wybieramy top-n unikalnych produktów z listy kandydatów

def recommend(
    basket: set[str],
    products: list[str],
    supports: Supports,
    n: int,
) -> list[str]:
    """Zwraca top-n rekomendacji na podstawie analizy koszykowej."""
    candidates = generate_next_product_candidates(basket, products, supports)
    return [candidate.item for candidate in candidates[:n]]

In [93]:
print("Koszyk:", baskets[1])
print("Rekomendacje:", recommend(baskets[1], products, supports, N))

Koszyk: {'sausage', 'semi-finished bread', 'yogurt', 'whole milk'}
Rekomendacje: ['rolls/buns', 'rolls/buns', 'soda', 'soda', 'bottled beer', 'citrus fruit', 'pastry', 'curd', 'frozen vegetables', 'beverages', 'frozen meals', 'sliced cheese', 'grapes']


In [94]:
print("Koszyk:", baskets[33])
print("Rekomendacje:", recommend(baskets[33], products, supports, N))

Koszyk: {'soda', 'domestic eggs', 'tropical fruit', 'photo/film', 'white wine', 'yogurt', 'root vegetables'}
Rekomendacje: ['sausage', 'sausage', 'citrus fruit', 'shopping bags', 'bottled beer', 'newspapers', 'coffee', 'frozen vegetables', 'white bread', 'uht-milk', 'beverages', 'meat', 'sliced cheese', 'grapes', 'grapes']


In [71]:
def test_one_basket_support(basket, baskets):
    count = 0
    for other_basket in baskets:
        if basket.issubset(other_basket):
            count += 1
    return count / len(baskets)

print(test_one_basket_support(baskets[33], baskets))

6.683151774376797e-05


## Część 4. - ewaluacja rekomendacji: MRR@k

### Metryka

**MRR@k** (Mean Reciprocal Rank) mierzy, na jakiej *pozycji* pojawił się ukryty produkt wśród top-k rekomendacji. Nagradza wyższe miejsca na liście:

$$\text{MRR@k} = \frac{1}{|\mathcal{Q}|} \sum_{q \in \mathcal{Q}} \frac{1}{\text{rank}_q}$$

gdzie:
- $\mathcal{Q}$ to zbiór przypadków testowych
- $\text{rank}_q$ to pozycja pierwszego trafnego produktu w top-k dla danego przypadku testowego $q$ (lub $\infty$, jeśli nie znaleziono — wkład do sumy wynosi wtedy 0).

### Protokół ewaluacji

Dla każdego koszyka testowego:
1. losowo wydzielamy jeden ukryty produkt; ten produkt byłby tym, który użytkownik najchętniej umieści w koszyku i tym samym tym, który powinien zaproponować nasz algorytm,
2. pozostałe produkty stanowią kontekst (czyli koszyk) przekazywany rekomendatorowi,
3. sprawdzamy, na której pozycji top-k pojawia się ukryty produkt.

Porównamy dwa algorytmy:
- **basket** – rekomendator oparty na analizie koszykowej (confidence + lift),
- **random** – rekomendator losowy (baseline).

Porównanie możemy przeprowadzić dla różnych wartości $k$.

In [82]:
# algorytm losowy - punkt odniesienia (baseline)

def recommend_random(
    basket: set[str],
    products: list[str],
    k: int,
    rng: random.Random | None = None,
) -> list[str]:
    """Zwraca k losowo wybranych produktów spoza koszyka."""
    if rng is None:
        rng = random.Random(SEED)
    
    available_products = [product for product in products if product not in basket]
    return rng.sample(available_products, min(k, len(available_products)))

In [83]:
# implementacja metryki MRR@k

def reciprocal_rank_at_k(recommended: list[str], relevant: set[str], k: int) -> float:
    """Zwraca odwrotność pozycji pierwszego trafnego produktu w top-k (0 jeśli brak trafienia)."""
    for rank, product in enumerate(recommended[:k], start=1):
        if product in relevant:
            return 1 / rank
    return 0.0

In [88]:
# podział koszyka na kontekst i ukryte produkty (ground truth)

def split_basket(
    basket: set[str],
    sample_size: int,
    rng: random.Random,
) -> tuple[set[str], set[str]]:
    """Losowo wydziela `sample_size` produktów jako cel dla rekomendacji; reszta to koszyk"""
    selected = rng.sample(sorted(basket), min(sample_size, len(basket)))
    basket_remaining = basket - set(selected)
    return basket_remaining, set(selected)

In [89]:
# typy reprezentujące wyniki ewaluacji

class MetricAtK(NamedTuple):
    k: int
    mrr: float


class EvaluationResults(NamedTuple):
    basket: MetricAtK
    random: MetricAtK

In [90]:
# ewaluacja obu rekomendatorów dla zadanego k

def evaluate(
    baskets: list[set[str]],
    products: list[str],
    supports: Supports,
    k: int,
    n_eval: int = N_EVAL,
    seed: int = SEED,
) -> EvaluationResults:
    """
    Generuje `n_eval` przypadków testowych przy uzyciu funkcji `split_basket`.
    Dla kazdego przypadku testowego uruchamia oba algorytmy, a następnie oblicza `MMR@k`.
    Zwraca średnie wartości MMR@k dla wskazanego `k` i dla obu algorytmów.
    """
    basket_results = []
    random_results = []
    for n in range(n_eval):
        rng = random.Random(seed + n)
        basket = rng.choice(baskets)
        basket_remaining, relevant = split_basket(basket, sample_size=1, rng=rng)

        recommended_basket = recommend(basket_remaining, products, supports, n=k)
        recommended_random = recommend_random(basket_remaining, products, k=k, rng=rng)

        mrr_basket = reciprocal_rank_at_k(recommended_basket, relevant, k)
        mrr_random = reciprocal_rank_at_k(recommended_random, relevant, k)
        basket_results.append(mrr_basket)
        random_results.append(mrr_random)
    
    basket_avg = sum(basket_results) / len(basket_results)
    random_avg = sum(random_results) / len(random_results)
    return EvaluationResults(
        basket=MetricAtK(k=k, mrr=basket_avg),
        random=MetricAtK(k=k, mrr=random_avg),
    )

In [95]:
# wyświetlamy wyniki i porównujemy algorytmy dla każdego k z listy K

def print_evaluation(results: EvaluationResults) -> None:
    col_w = 10
    header = f"{'Algorytm':<12} | {'mrr@' + str(results.basket.k):>{col_w}}"
    print(header)
    print('-' * len(header))
    for algo, metric in results._asdict().items():
        print(f"{algo:<12} | {metric.mrr:>{col_w}.4f}")


for k in K:
    print_evaluation(evaluate(baskets, products, supports, k))
    print()

Algorytm     |      mrr@3
-------------------------
basket       |     0.0367
random       |     0.0042

Algorytm     |      mrr@5
-------------------------
basket       |     0.0397
random       |     0.0077

Algorytm     |     mrr@10
-------------------------
basket       |     0.0409
random       |     0.0117

